# AIOps W2D1 - Alert Correlator

Notebook chạy pipeline correlate cho `alerts_sample.jsonl` và `services.json`, sau đó ghi `results/cluster_summary.json`.

In [7]:
from pathlib import Path
import json

from correlate import correlate, fingerprint, session_groups, load_service_graph, topology_groups

BASE = Path('.')
DATASET = BASE / 'dataset'
RESULTS = BASE / 'results'
RESULTS.mkdir(exist_ok=True)

print(f'Working directory: {BASE.resolve()}')

Working directory: /Users/trandinhthong/Downloads/AIOps/w2/d1


In [3]:
alerts = [json.loads(line) for line in (DATASET / 'alerts_sample.jsonl').read_text().splitlines() if line.strip()]
services_doc = json.loads((DATASET / 'services.json').read_text())

print(f'Loaded alerts: {len(alerts)}')
print(f'Loaded services: {len(services_doc["services"])}')
print(f'Loaded edges: {len(services_doc["edges"])}')

Loaded alerts: 20
Loaded services: 10
Loaded edges: 17


In [4]:
sessions = session_groups(alerts, gap_sec=49)
print(f'Sessions with gap_sec=49: {len(sessions)}')
for idx, group in enumerate(sessions):
    print(idx, len(group), group[0]['ts'], '->', group[-1]['ts'])

Sessions with gap_sec=49: 1
0 20 2026-06-12T09:42:01Z -> 2026-06-12T09:48:30Z


In [5]:
summary = correlate(alerts, services_doc, gap_sec=49, max_hop=2)
outfile = RESULTS / 'cluster_summary.json'
outfile.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(f'Wrote {outfile}')
print(f"input_alerts={summary['input_alerts']} output_clusters={summary['output_clusters']} reduction_ratio={summary['reduction_ratio']}")

Wrote results/cluster_summary.json
input_alerts=20 output_clusters=5 reduction_ratio=0.75


In [ ]:
for cluster in summary['clusters']:
    print(cluster['cluster_id'], cluster['alert_count'], cluster['services'], cluster['time_range'], cluster['max_severity'])